In [ ]:
from google.colab import drive


drive.mount('/content/drive')

Mounted at /content/drive


#Train + Evaluation

In [ ]:
import json
import torch
import numpy as np
import pandas as pd
import os
import shutil
import gc
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, set_seed
from sklearn.metrics import mean_squared_error
from google.colab import drive

# --- 1. הגדרות בסיס ---
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/SemEval"
train_path = f"{base_path}/train.json"
dev_path = f"{base_path}/dev.json"

# תיקיית פלט ראשית לשלב הולידציה
output_phase1 = f"{base_path}/hybrid_phase1_validation"
if os.path.exists(output_phase1): shutil.rmtree(output_phase1)
os.makedirs(output_phase1, exist_ok=True)

# הגדרת המודלים
MODELS = {
    "standard": "microsoft/deberta-v3-large",
    "nli": "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
}

MIN_SCORE, MAX_SCORE = 1.0, 5.0
LR_RATE = 1e-5
set_seed(42)

# --- 2. מחלקות Dataset ---
def load_data(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        if isinstance(data, dict): return list(data.values())
        return data

# Dataset למודל הרגיל
class StandardDataset(Dataset):
    def __init__(self, data_list, tokenizer, max_len=512):
        self.data = data_list
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.data)
    def __getitem__(self, index):
        item = self.data[index]
        homonym = item.get('homonym', '')
        sentence = item['sentence']
        if homonym and homonym in sentence:
            highlighted_sentence = sentence.replace(homonym, f'" {homonym} "')
        else: highlighted_sentence = sentence
        full_story = f"{item['precontext']} {highlighted_sentence} {item['ending']}"
        normalized = (item['average'] - MIN_SCORE) / (MAX_SCORE - MIN_SCORE)
        inputs = self.tokenizer(item['judged_meaning'], full_story, max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': inputs['input_ids'].flatten(), 'attention_mask': inputs['attention_mask'].flatten(), 'labels': torch.tensor(float(normalized), dtype=torch.float)}

# Dataset למודל NLI
class NLIDataset(Dataset):
    def __init__(self, data_list, tokenizer, max_len=512):
        self.data = data_list
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.data)
    def __getitem__(self, index):
        item = self.data[index]
        homonym = item.get('homonym', '')
        sentence = item['sentence']
        if homonym and homonym in sentence:
            highlighted_sentence = sentence.replace(homonym, f'" {homonym} "')
        else: highlighted_sentence = sentence
        premise = f"{item['precontext']} {highlighted_sentence} {item['ending']}"
        hypothesis = f'The word "{homonym}" implies: {item["judged_meaning"]}'
        normalized = (item['average'] - MIN_SCORE) / (MAX_SCORE - MIN_SCORE)
        inputs = self.tokenizer(premise, hypothesis, max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': inputs['input_ids'].flatten(), 'attention_mask': inputs['attention_mask'].flatten(), 'labels': torch.tensor(float(normalized), dtype=torch.float)}

# --- 3. מנוע האימון והשמירה ---
def run_validation_cycle(model_type, model_name, train_data, dev_data):
    print(f"\n🚀 STARTING PHASE 1 for: {model_type.upper()}")

    # הגדרת נתיבים
    model_save_path = f"{output_phase1}/model_{model_type}"
    preds_save_path = f"{output_phase1}/preds_{model_type}.jsonl"

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    if model_type == "standard":
        train_ds = StandardDataset(train_data, tokenizer)
        dev_ds = StandardDataset(dev_data, tokenizer)
    else:
        train_ds = NLIDataset(train_data, tokenizer)
        dev_ds = NLIDataset(dev_data, tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=1, problem_type="regression", ignore_mismatched_sizes=True
    )

    training_args = TrainingArguments(
        output_dir=model_save_path,
        num_train_epochs=5,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=LR_RATE,
        weight_decay=0.01,
        save_strategy="no",
        fp16=True,
        report_to="none"
    )

    trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, tokenizer=tokenizer)

    print(f"   🔥 Training on TRAIN...")
    trainer.train()

    # 1. שמירת המודל
    print(f"   💾 Saving Model to: {model_save_path}")
    trainer.save_model(model_save_path)
    tokenizer.save_pretrained(model_save_path)

    # 2. חיזוי ושמירה
    print(f"   🧪 Predicting on DEV...")
    output = trainer.predict(dev_ds)
    logits = output.predictions[0] if isinstance(output.predictions, tuple) else output.predictions
    preds = logits.flatten() * (MAX_SCORE - MIN_SCORE) + MIN_SCORE
    preds = np.clip(preds, 1.0, 5.0)

    dev_ids = [x.get('id', str(i)) for i, x in enumerate(dev_data)]
    results = [{"id": pid, "prediction": float(p)} for pid, p in zip(dev_ids, preds)]

    with open(preds_save_path, 'w') as f:
        for entry in results: json.dump(entry, f); f.write('\n')
    print(f"   📝 Saved Predictions to: {preds_save_path}")

    del model; del trainer; del tokenizer; gc.collect(); torch.cuda.empty_cache()
    return preds

# --- 4. Main Execution Phase 1 ---
train_raw = load_data(train_path)
dev_raw = load_data(dev_path)

# הרצת המודלים
preds_std = run_validation_cycle("standard", MODELS["standard"], train_raw, dev_raw)
preds_nli = run_validation_cycle("nli", MODELS["nli"], train_raw, dev_raw)

# --- 5. דוח סיכום ואנסמבל ---
print("\n" + "="*50)
print("📊 PHASE 1 REPORT (VALIDATION)")
print("="*50)

def print_metrics(preds, name):
    actuals = [x['average'] for x in dev_raw]
    stds = [x.get('stdev', x.get('std', 0.5)) for x in dev_raw]
    df = pd.DataFrame({'Actual': actuals, 'Predicted': preds, 'Stdev': stds})
    df['Error'] = abs(df['Actual'] - df['Predicted'])
    df['Threshold'] = df['Stdev'].apply(lambda x: max(x, 1.0))
    df['Within_SD'] = df['Error'] <= df['Threshold']
    mse = mean_squared_error(actuals, preds)
    acc = df['Within_SD'].mean()
    print(f"🔹 {name:<15} | MSE: {mse:.4f} | Accuracy: {acc:.2%}")

# חישוב אנסמבל
preds_ensemble = (preds_std + preds_nli) / 2

# הדפסת דוח
print_metrics(preds_std, "Standard")
print_metrics(preds_nli, "NLI Expert")
print("-" * 50)
print_metrics(preds_ensemble, "HYBRID ENSEMBLE")

# שמירת האנסמבל
ensemble_path = f"{output_phase1}/preds_HYBRID_ENSEMBLE.jsonl"
dev_ids = [x.get('id', str(i)) for i, x in enumerate(dev_raw)]
results = [{"id": pid, "prediction": float(p)} for pid, p in zip(dev_ids, preds_ensemble)]
with open(ensemble_path, 'w') as f:
    for entry in results: json.dump(entry, f); f.write('\n')

print(f"\n✅ Validations saved in: {output_phase1}")


🚀 STARTING PHASE 1 for: STANDARD


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2444581643.py:115: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `proces

   🔥 Training on TRAIN...


Step,Training Loss
500,0.063600


   💾 Saving Model to: /content/drive/MyDrive/SemEval/hybrid_phase1_validation/model_standard
   🧪 Predicting on DEV...


...

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([3, 1024]) in the checkpoint and torch.Size([1, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2444581643.py:115: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, tokenizer=tokenizer)
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, be

   🔥 Training on TRAIN...


Step,Training Loss
500,0.048700


   💾 Saving Model to: /content/drive/MyDrive/SemEval/hybrid_phase1_validation/model_nli
   🧪 Predicting on DEV...


   📝 Saved Predictions to: /content/drive/MyDrive/SemEval/hybrid_phase1_validation/preds_nli.jsonl

📊 PHASE 1 REPORT (VALIDATION)
🔹 Standard        | MSE: 1.0453 | Accuracy: 75.00%
🔹 NLI Expert      | MSE: 0.9428 | Accuracy: 77.04%
--------------------------------------------------
🔹 HYBRID ENSEMBLE | MSE: 0.8897 | Accuracy: 79.42%

✅ Validations saved in: /content/drive/MyDrive/SemEval/hybrid_phase1_validation


#Test

In [ ]:
import json
import torch
import numpy as np
import os
import shutil
import gc
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, set_seed
from google.colab import drive

# --- הגדרות PHASE 2 ---
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/SemEval"
train_path = f"{base_path}/train.json"
dev_path = f"{base_path}/dev.json"
test_path = f"{base_path}/test.json"
output_phase2 = f"{base_path}/hybrid_phase2_submission"

MODELS = {
    "standard": "microsoft/deberta-v3-large",
    "nli": "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
}

MIN_SCORE, MAX_SCORE = 1.0, 5.0
LR_RATE = 1e-5
set_seed(42)

if os.path.exists(output_phase2): shutil.rmtree(output_phase2)
os.makedirs(output_phase2, exist_ok=True)

def load_data(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        if isinstance(data, dict): return list(data.values())
        return data

# --- Datasets מתוקנים (מטפלים בנתוני הטסט החסרים) ---

class StandardDataset(Dataset):
    def __init__(self, data_list, tokenizer, max_len=512):
        self.data = data_list
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.data)
    def __getitem__(self, index):
        item = self.data[index]
        homonym = item.get('homonym', '')
        sentence = item['sentence']
        if homonym and homonym in sentence:
            highlighted_sentence = sentence.replace(homonym, f'" {homonym} "')
        else: highlighted_sentence = sentence
        full_story = f"{item['precontext']} {highlighted_sentence} {item['ending']}"

        # --- התיקון כאן: טיפול בערכים חסרים בטסט ---
        try:
            raw_val = item.get('average')
            # מנסים להמיר למספר. אם זה "(???)" זה יקפוץ ל-except
            val = float(raw_val)
            normalized = (val - MIN_SCORE) / (MAX_SCORE - MIN_SCORE)
        except (ValueError, TypeError):
            normalized = 0.0 # ברירת מחדל לטסט (לא משפיע על החיזוי)

        inputs = self.tokenizer(item['judged_meaning'], full_story, max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': inputs['input_ids'].flatten(), 'attention_mask': inputs['attention_mask'].flatten(), 'labels': torch.tensor(float(normalized), dtype=torch.float)}

class NLIDataset(Dataset):
    def __init__(self, data_list, tokenizer, max_len=512):
        self.data = data_list
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.data)
    def __getitem__(self, index):
        item = self.data[index]
        homonym = item.get('homonym', '')
        sentence = item['sentence']
        if homonym and homonym in sentence:
            highlighted_sentence = sentence.replace(homonym, f'" {homonym} "')
        else: highlighted_sentence = sentence
        premise = f"{item['precontext']} {highlighted_sentence} {item['ending']}"
        hypothesis = f'The word "{homonym}" implies: {item["judged_meaning"]}'

        # --- התיקון כאן: טיפול בערכים חסרים בטסט ---
        try:
            raw_val = item.get('average')
            val = float(raw_val)
            normalized = (val - MIN_SCORE) / (MAX_SCORE - MIN_SCORE)
        except (ValueError, TypeError):
            normalized = 0.0

        inputs = self.tokenizer(premise, hypothesis, max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': inputs['input_ids'].flatten(), 'attention_mask': inputs['attention_mask'].flatten(), 'labels': torch.tensor(float(normalized), dtype=torch.float)}

# --- מנוע אימון וחיזוי לטסט ---
def run_test_cycle(model_type, model_name, full_data, test_data_values, test_ids):
    print(f"\n🚀 STARTING PHASE 2 for: {model_type.upper()}")

    model_save_path = f"{output_phase2}/final_model_{model_type}"
    preds_save_path = f"{output_phase2}/submission_{model_type}.jsonl"

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Dataset Selection
    if model_type == "standard":
        full_ds = StandardDataset(full_data, tokenizer)
        test_ds = StandardDataset(test_data_values, tokenizer)
    else:
        full_ds = NLIDataset(full_data, tokenizer)
        test_ds = NLIDataset(test_data_values, tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=1, problem_type="regression", ignore_mismatched_sizes=True
    )

    training_args = TrainingArguments(
        output_dir=model_save_path,
        num_train_epochs=5,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=LR_RATE,
        weight_decay=0.01,
        save_strategy="no",
        fp16=True,
        report_to="none"
    )

    trainer = Trainer(model=model, args=training_args, train_dataset=full_ds, tokenizer=tokenizer)

    print(f"   🔥 Training on ALL DATA ({len(full_data)} samples)...")
    trainer.train()

    # 1. שמירת המודל הסופי
    print(f"   💾 Saving Final Model to: {model_save_path}")
    trainer.save_model(model_save_path)
    tokenizer.save_pretrained(model_save_path)

    # 2. חיזוי על TEST
    print(f"   🔮 Predicting on TEST...")
    output = trainer.predict(test_ds)
    logits = output.predictions[0] if isinstance(output.predictions, tuple) else output.predictions
    preds = logits.flatten() * (MAX_SCORE - MIN_SCORE) + MIN_SCORE
    preds = np.clip(preds, 1.0, 5.0)

    # שמירה
    results = [{"id": pid, "prediction": float(p)} for pid, p in zip(test_ids, preds)]
    with open(preds_save_path, 'w') as f:
        for entry in results: json.dump(entry, f); f.write('\n')

    print(f"   📝 Saved Submission to: {preds_save_path}")

    del model; del trainer; del tokenizer; gc.collect(); torch.cuda.empty_cache()
    return preds

# --- Main Execution Phase 2 ---
print("📥 Merging Data for Phase 2...")
train_raw = load_data(train_path)
dev_raw = load_data(dev_path)
full_data = train_raw + dev_raw

# טעינת הטסט וסידור ה-IDs
with open(test_path, 'r', encoding='utf-8') as f:
    raw_test_dict = json.load(f)
# שימוש במפתחות כמזהים (0, 1, 2...)
test_ids = list(raw_test_dict.keys())
test_data_values = list(raw_test_dict.values())

# הרצת המודלים
preds_std_test = run_test_cycle("standard", MODELS["standard"], full_data, test_data_values, test_ids)
preds_nli_test = run_test_cycle("nli", MODELS["nli"], full_data, test_data_values, test_ids)

# --- יצירת האנסמבל הסופי ---
print("\n⚖️ CALCULATING FINAL ENSEMBLE...")
final_ensemble_preds = (preds_std_test + preds_nli_test) / 2

final_submission_path = f"{output_phase2}/submission_HYBRID_FINAL.jsonl"
results = [{"id": pid, "prediction": float(p)} for pid, p in zip(test_ids, final_ensemble_preds)]

with open(final_submission_path, 'w', encoding='utf-8') as f:
    for entry in results: json.dump(entry, f); f.write('\n')

# העתקה גם לתיקייה הראשית לנוחות
shutil.copy(final_submission_path, f"{base_path}/submission_HYBRID_FINAL.jsonl")

print("\n" + "="*50)
print(f"✅ MISSION COMPLETE!")
print(f"📂 Final Models saved in: {output_phase2}")
print(f"📄 Final Submission: {final_submission_path}")
print("="*50)

📥 Merging Data for Phase 2...

🚀 STARTING PHASE 2 for: STANDARD


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3835871391.py:128: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `proces

   🔥 Training on ALL DATA (2868 samples)...


Step,Training Loss
500,0.064700


   💾 Saving Final Model to: /content/drive/MyDrive/SemEval/hybrid_phase2_submission/final_model_standard
   🔮 Predicting on TEST...


...

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([3, 1024]) in the checkpoint and torch.Size([1, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3835871391.py:128: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, args=training_args, train_dataset=full_ds, tokenizer=tokenizer)
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, bei

   🔥 Training on ALL DATA (2868 samples)...


Step,Training Loss
500,0.052300


   💾 Saving Final Model to: /content/drive/MyDrive/SemEval/hybrid_phase2_submission/final_model_nli
   🔮 Predicting on TEST...


   📝 Saved Submission to: /content/drive/MyDrive/SemEval/hybrid_phase2_submission/submission_nli.jsonl

⚖️ CALCULATING FINAL ENSEMBLE...

✅ MISSION COMPLETE!
📂 Final Models saved in: /content/drive/MyDrive/SemEval/hybrid_phase2_submission
📄 Final Submission: /content/drive/MyDrive/SemEval/hybrid_phase2_submission/submission_HYBRID_FINAL.jsonl


#Spearman

In [ ]:
import json
import os
import numpy as np
from scipy.stats import spearmanr
from google.colab import drive

# --- 1. הגדרות וחיבור ---
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/SemEval"
dev_path = f"{base_path}/dev.json"

# התיקייה שבה שמרנו את תוצאות Phase 1 (וודאי שזה הנתיב הנכון מהריצה הקודמת)
predictions_dir = f"{base_path}/hybrid_phase1_validation"

# רשימת הקבצים שאנחנו רוצים לבדוק
files_to_check = [
    "preds_standard.jsonl",
    "preds_nli.jsonl",
    "preds_HYBRID_ENSEMBLE.jsonl"
]

print(f"📊 CALCULATING SPEARMAN CORRELATION (Offline)")
print(f"📂 Reading from: {predictions_dir}")
print("-" * 50)

# --- 2. טעינת האמת (Ground Truth) ---
with open(dev_path, 'r', encoding='utf-8') as f:
    dev_raw = json.load(f)
    if isinstance(dev_raw, dict): dev_data = list(dev_raw.values())
    else: dev_data = dev_raw

# יצירת מילון {id: score} לאמת - כדי להבטיח סנכרון מושלם
# מניחים שה-ID הוא המפתח או שדה 'id'
true_scores_map = {}
for i, item in enumerate(dev_data):
    # מנסים למצוא ID, אם אין משתמשים באינדקס כמו שעשינו בייצור
    pid = item.get('id', str(i))
    true_scores_map[str(pid)] = float(item['average'])

# --- 3. לולאה על קבצי הפרדיקציה ---
for filename in files_to_check:
    file_path = f"{predictions_dir}/{filename}"

    if not os.path.exists(file_path):
        print(f"❌ File not found: {filename}")
        continue

    # טעינת הפרדיקציות
    preds_map = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            entry = json.loads(line)
            preds_map[str(entry['id'])] = float(entry['prediction'])

    # --- 4. סנכרון וחישוב ---
    # אנחנו ניקח רק IDs שקיימים גם באמת וגם בפרדיקציה (אמור להיות כולם)
    common_ids = [pid for pid in true_scores_map.keys() if pid in preds_map]

    if len(common_ids) == 0:
        print(f"⚠️ Critical Error: No matching IDs found for {filename}")
        continue

    y_true = [true_scores_map[pid] for pid in common_ids]
    y_pred = [preds_map[pid] for pid in common_ids]

    # חישוב ספירמן
    spearman_score, p_value = spearmanr(y_true, y_pred)

    # הצגה יפה
    clean_name = filename.replace("preds_", "").replace(".jsonl", "").replace("_", " ")
    print(f"🔹 {clean_name:<20}")
    print(f"   Spearman: {spearman_score:.4f}")
    # אופציונלי: להציג גם P-value אם רוצים לדעת מובהקות
    # print(f"   P-value:  {p_value:.4e}")
    print("-" * 30)

print("✅ Done.")

📊 CALCULATING SPEARMAN CORRELATION (Offline)
📂 Reading from: /content/drive/MyDrive/SemEval/hybrid_phase1_validation
--------------------------------------------------
🔹 standard            
   Spearman: 0.5873
------------------------------
🔹 nli                 
   Spearman: 0.6636
------------------------------
🔹 HYBRID ENSEMBLE     
   Spearman: 0.6663
------------------------------
✅ Done.
